<a href="https://colab.research.google.com/github/Abhi712-ui/lis501-ipynb-files/blob/main/LIS_501_(Week_2_Basics_of_Processing_Text_Data).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LIS 501 · Week 2 - Basics of Processing Text Data


**Guiding question:** What does a processing decision preserve, combine, or remove from our evidence?

By the end, you should be able to run and edit short Python cells, turn a string into a list of tokens, explain a normalization choice, and inspect a few linguistic annotations.


### Our route through the demo

| No. | Activity | Participation |
|:--|:--|:--|
| 1. | Open Colab and run setup | Follow along |
| 2. | Strings, lists, indexing, splitting, joining; initials warm-up | Guided coding |
| 3. | **Exercise 1: From a headline to tokens** | Work with a neighbor |
| 4. | Tokenization and sentence segmentation | Predict, run, inspect |
| 5. | Case-folding and stop-word removal | Guided coding |
| 6. | **Exercise 2: Preserve the meaning** | Work with a neighbor |
| 7. | Stemming and lemmatization | Instructor-led preview |
| 8. | POS tags, noun phrases, and named entities | Instructor-led preview |
| 9. | Connect one choice to your project | Brief discussion |




## **0. Start in Colab**

Open Google Colab and use **File → Upload notebook** to open this `.ipynb` file. Use the default CPU runtime; this demo does not need a GPU, an API key, or a data-file upload. [Colab help](https://research.google.com/colaboratory/faq.html)

A **markdown cell** contains explanations like this one. A **code cell** contains instructions Python can execute. Click the play button to run a cell, or use **Shift + Enter** to run it and move down.

Run cells **from top to bottom**, one at a time during the demo. Variables created in one cell are available in later cells. After a runtime restart, run the earlier cells again. The unfinished exercise cells are safe to run; their checks will initially report `False`.

### Setup: run these two cells as supplied
The cells install the versions used for this notebook. The first installs two text-processing libraries; the second installs the small English model used near the end. Internet access is needed for installation. You do not need to learn the setup code today. If Colab requests a runtime restart, restart and rerun these setup cells before continuing. [Model installation](https://spacy.io/usage/models#download)


In [ ]:
#@title Setup A: install the text-processing libraries
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
     "nltk==3.9.2", "spacy==3.8.11"],
    check=True,
)
print("Library installation finished. Next, run Setup B.")


Library installation finished. Next, run Setup B.


In [ ]:
#@title Setup B: install the small English model
import subprocess
import sys

model_url = (
    "https://github.com/explosion/spacy-models/releases/download/"
    "en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl"
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
     "--no-deps", model_url],
    check=True,
)
print("Model installation finished. We will load it in Section 6.")


Model installation finished. We will load it in Section 6.


## **1. Python string and text basics**

### 1.1 Store text and display it
A **string** is text inside quotation marks. A **variable** is a name we give a value. The `=` below assigns the string to `text`. `print()` displays a value. A line beginning with `#` is a comment for the reader.

Single and double quotation marks both work. Use matching quotation marks at the beginning and end.


In [ ]:
# Keep the welcome example from the original notebook.
text = "Welcome to LIS 501!"
print(text)


Welcome to LIS 501!


### 1.2 A string is a sequence of characters
`len()` counts the characters, including spaces and punctuation. `list()` makes those individual characters visible in a list. A **list** is an ordered collection shown inside square brackets.

**Predict:** Will the spaces and the exclamation mark be included in the count?


In [ ]:
print("Number of characters:", len(text))
characters = list(text)
print(characters)


Number of characters: 19
['W', 'e', 'l', 'c', 'o', 'm', 'e', ' ', 't', 'o', ' ', 'L', 'I', 'S', ' ', '5', '0', '1', '!']


### 1.3 Apply an operation to a string
A **method** is an operation attached to a value, such as `text.lower()`. The dot selects the method; the parentheses run it.

These methods return new strings. Store the result under another name to preserve the original. Notice how title case and capitalization also affect the abbreviation `LIS`. [Python string methods](https://docs.python.org/3/library/stdtypes.html#string-methods)


In [ ]:
lowercase_text = text.lower()
print("Lowercase:", lowercase_text)
print("Uppercase:", text.upper())
print("Title case:", text.title())
print("Capitalized:", text.capitalize())
print("Original:", text)


Lowercase: welcome to lis 501!
Uppercase: WELCOME TO LIS 501!
Title case: Welcome To Lis 501!
Capitalized: Welcome to lis 501!
Original: Welcome to LIS 501!


### 1.4 Select characters with indexing and slicing
Python starts counting positions at **0**. An **index** selects one item; a **slice** selects a range. In `[start:stop]`, the start is included and the stop is excluded. A negative index counts back from the end.

**Predict:** Which character will `text[1]` return? [Python introduction](https://docs.python.org/3/tutorial/introduction.html#text)


In [ ]:
print("First character:", text[0])
print("Second character:", text[1])
print("First three characters:", text[0:3])
print("Last character:", text[-1])
print("Everything except the last three:", text[0:-3])


First character: W
Second character: e
First three characters: Wel
Last character: !
Everything except the last three: Welcome to LIS 5


### 1.5 Split a string; join a list
`split()` with no argument separates a string at whitespace and returns a list. You can index that list just as you indexed the string. `join()` combines a list of strings using a chosen separator.

**Check the unit:** `len(text)` counts characters; `len(words)` counts items in the list. [Python string methods](https://docs.python.org/3/library/stdtypes.html#string-methods)


In [ ]:
words = text.split()
print("List:", words)
print("Number of list items:", len(words))
print("First item:", words[0])
print("Joined with spaces:", " ".join(words))
print("Joined with slashes:", " / ".join(words))


List: ['Welcome', 'to', 'LIS', '501!']
Number of list items: 4
First item: Welcome
Joined with spaces: Welcome to LIS 501!
Joined with slashes: Welcome / to / LIS / 501!


### 1.6 Warm-up Exercise: first and last initials
We want to get the initials of a person's name. We split the name, select a character from each selected name part, and combine the characters with `+`.

Read `name_parts[0][0]` in two steps: take the first item in the list, then take the first character of that string. This example assumes a two-part name; real naming conventions require more care.


In [ ]:
name = "Peter Pan"
name_parts = name.split()
print("Name parts:", name_parts)

first_initial = name_parts[0][0]
last_initial = name_parts[-1][0]
initials = first_initial + last_initial
print("Initials:", initials.upper())


Name parts: ['Peter', 'Pan']
Initials: PP


## **2. In-class exercise 1: Headline to tokens · 10 minutes**

**Scenario:** You are preparing a small collection of headlines for a word-frequency comparison.

Work with a neighbor. Complete the starter cell by replacing the values on the lines marked `TODO`. Keep the variable names. Create a lowercased copy with surrounding whitespace removed, split it into a list, and report the list's length and its first and last items. Preserve `headline` unchanged.

**New method:** `.strip()` removes whitespace from the beginning and end. It leaves spaces inside the string. You can chain operations, for example `some_text.strip().lower()`.

For the supplied headline, your tokens should be:
```text
['students', 'study', 'climate', 'reports.']
```
The period remains attached to `reports.`. We will investigate punctuation in the next section.

**Before moving on:** Explain to your neighbor why the original string and the token list have different lengths.


In [ ]:
headline = "   STUDENTS study climate reports.   "

# TODO: remove surrounding whitespace and lowercase a COPY.
normalized_headline = headline

# TODO: split that copy into a list.
headline_tokens = []

# TODO: calculate the list's length and select its first and last items.
token_count = 0
first_token = ""
last_token = ""

print("Original:", headline)
print("Normalized:", normalized_headline)
print("Tokens:", headline_tokens)
print("Count:", token_count)
print("First and last:", first_token, last_token)


Original:    STUDENTS study climate reports.   
Normalized:    STUDENTS study climate reports.   
Tokens: []
Count: 0
First and last:  


### Check your result
Run the supplied checks. `==` asks whether two values are equal. `True` means a check passed; `False` means you should revisit that part. You do not need to edit this cell.


In [ ]:
print("Original preserved:", headline == "   STUDENTS study climate reports.   ")
print("Normalized copy correct:", normalized_headline == "students study climate reports.")
print("Token list correct:", headline_tokens == ["students", "study", "climate", "reports."])
print("Token count correct:", token_count == 4)
print("First token correct:", first_token == "students")
print("Last token correct:", last_token == "reports.")


Original preserved: True
Normalized copy correct: False
Token list correct: False
Token count correct: False
First token correct: False
Last token correct: False


## **3. Tokenization and sentence segmentation**
*Lecture connections: slides 6–17.*

### 3.1 Whitespace splitting and punctuation replacement
Return to the parking example from the lecture. First split on whitespace. Then replace selected punctuation with spaces and split again.

The **`for` loop** repeats the indented instruction once for each item in the list. `symbol` takes each value in turn. `.replace(old, new)` returns a string with the specified replacement. We repeatedly update a working copy and keep `parking` unchanged.

**Predict:** What will happen to the price and the institution's name?


In [ ]:
parking = "Parking is $4.50/hour at UW-Madison."
print("Whitespace only:", parking.split())

punctuation_replaced = parking
for symbol in ["$", ".", "/", "-"]:
    punctuation_replaced = punctuation_replaced.replace(symbol, " ")

print("Punctuation replaced:", punctuation_replaced.split())
print("Original:", parking)


Whitespace only: ['Parking', 'is', '$4.50/hour', 'at', 'UW-Madison.']
Punctuation replaced: ['Parking', 'is', '4', '50', 'hour', 'at', 'UW', 'Madison']
Original: Parking is $4.50/hour at UW-Madison.


### 3.2 Use an established tokenizer
`import` gives us access to code from a library. **NLTK** supplies the Treebank tokenizer used here. It applies tokenization rules without needing a downloaded training corpus.

Compare its output with the previous cell. The decimal stays together, the dollar sign becomes a separate token, and this tokenizer keeps `4.50/hour` as one token. That last decision might need further processing for a study of prices. [NLTK Treebank tokenizer](https://www.nltk.org/api/nltk.tokenize.TreebankWordTokenizer.html)


In [ ]:
from nltk.tokenize import TreebankWordTokenizer

tokenizer = TreebankWordTokenizer()
print("Treebank:", tokenizer.tokenize(parking))
print("Lecture example:", tokenizer.tokenize("That U.S.A. poster-print costs $12.40..."))


Treebank: ['Parking', 'is', '$', '4.50/hour', 'at', 'UW-Madison', '.']
Lecture example: ['That', 'U.S.A.', 'poster-print', 'costs', '$', '12.40', '...']


### 3.3 Recognize sentence boundaries
A period can appear inside an abbreviation. Compare splitting at every period with an English tokenizer plus a simple sentence-boundary component.

We use **spaCy**, another text-processing library. `spacy.blank("en")` supplies English tokenization rules; adding a `sentencizer` supplies rule-based sentence boundaries. This example uses **no trained model**. Its rules work on this paragraph; inspect them on your own text. [spaCy Sentencizer](https://spacy.io/api/sentencizer)


In [ ]:
import spacy

paragraph = (
    "The US is a big nation. Americans love the U.S.A. a lot. "
    "They like to drive their cars around the country. "
    "They measure speed in m.p.h and not km.p.h."
)
print("Split at every period:", paragraph.split("."))

sentence_rules = spacy.blank("en")
sentence_rules.add_pipe("sentencizer")
sentence_doc = sentence_rules(paragraph)

print("\nSentence boundaries from the rules:")
for sentence in sentence_doc.sents:
    print(sentence.text)


Split at every period: ['The US is a big nation', ' Americans love the U', 'S', 'A', ' a lot', ' They like to drive their cars around the country', ' They measure speed in m', 'p', 'h and not km', 'p', 'h', '']

Sentence boundaries from the rules:
The US is a big nation.
Americans love the U.S.A. a lot.
They like to drive their cars around the country.
They measure speed in m.p.h and not km.p.h.


## **4. Case-folding and stop-word removal**
*Lecture connections: slides 8–9 and 18–19.*

### 4.1 Standardize selected forms; preserve the source
The lecture maps `U.S.A.` to `US` for a country-mention count. We can express that specific mapping with `.replace()`. Lowercasing makes a different change: it also makes `US` and `us` look alike.

Here, we use `.lower()` for the English lowercasing operation illustrated in the lecture. Python also has a distinct `.casefold()` method for broader Unicode case-insensitive matching. [Python string methods](https://docs.python.org/3/library/stdtypes.html#string-methods)


In [ ]:
country_text = "The US and the U.S.A. appear in this example."
standardized_country_text = country_text.replace("U.S.A.", "US")
print("Specific mapping:", standardized_country_text)

case_example = "The US asked us to cut EMISSIONS."
print("Original:", case_example)
print("Lowercase copy:", case_example.lower())


Specific mapping: The US and the US appear in this example.
Original: The US asked us to cut EMISSIONS.
Lowercase copy: the us asked us to cut emissions.


### 4.2 Filter a small, explicit stop-word list
A stop-word list specifies which tokens to exclude for a particular analysis. The list below is a small teaching example.

An **`if` statement** applies its indented instruction only when the condition is true. `not in` tests whether a token is absent from the list. `.append()` adds an item to our growing result list.

Punctuation remains in this example. Excluding punctuation would be an additional decision. [Python loops and conditions](https://docs.python.org/3/tutorial/controlflow.html)


In [ ]:
stop_words = ["the", "and", "is", "at"]
original_sentence = "The little mouse ate the fresh cheese."
word_tokens = tokenizer.tokenize(original_sentence.lower())

kept_tokens = []
for token in word_tokens:
    if token not in stop_words:
        kept_tokens.append(token)

print("Before:", word_tokens)
print("After:", kept_tokens)


Before: ['the', 'little', 'mouse', 'ate', 'the', 'fresh', 'cheese', '.']
After: ['little', 'mouse', 'ate', 'fresh', 'cheese', '.']


## **5. In-class exercise 2: Preserve the meaning · 5 minutes**

**Scenario:** You are studying how sustainability reports frame commitments to reduce emissions. These two statements differ by one important word:

> The company will reduce emissions.  
> The company will **not** reduce emissions.

Run the starter cell once. It uses a proposed stop-word list that makes the two processed statements identical. **Edit only the stop-word list to preserve the difference**, then rerun the cell. Keep `the` and `will` excluded for this exercise. Add a short explanation in `decision_note`.

The filtering code is supplied. The outer loop processes one comment at a time; the inner loop considers each token in that comment. The period is kept in both outputs.

**Discuss with your neighbor:** Which word did you protect? How would losing it affect the story you could tell about these statements?


In [ ]:
comments = [
    "The company will reduce emissions.",
    "The company will not reduce emissions.",
]

# TODO: revise this list, then rerun the whole cell.
exercise_stop_words = ["the", "will", "not"]

# TODO: explain which distinction your choice preserves.
decision_note = ""

# Supplied filtering code: no edits needed below this line.
filtered_comments = []
for comment in comments:
    comment_tokens = tokenizer.tokenize(comment.lower())
    retained_tokens = []
    for token in comment_tokens:
        if token not in exercise_stop_words:
            retained_tokens.append(token)
    filtered_comments.append(retained_tokens)
    print("Original:", comment)
    print("Processed:", retained_tokens)

print("My reason:", decision_note)


Original: The company will reduce emissions.
Processed: ['company', 'reduce', 'emissions', '.']
Original: The company will not reduce emissions.
Processed: ['company', 'reduce', 'emissions', '.']
My reason: 


### Check your result
Here `in` tests for membership, `not in` tests for absence, and `!=` means “is different from.” A nonempty explanation is checked for completion, not for the quality of your reasoning.


In [ ]:
print("Statements remain different:", filtered_comments[0] != filtered_comments[1])
print("Negation preserved:", "not" in filtered_comments[1])
print("The excluded:", "the" not in filtered_comments[0] and "the" not in filtered_comments[1])
print("Will excluded:", "will" not in filtered_comments[0] and "will" not in filtered_comments[1])
print("Explanation added:", len(decision_note.strip()) > 0)


Statements remain different: False
Negation preserved: False
The excluded: True
Will excluded: True
Explanation added: False


## **6. Stemming and lemmatization**
*Lecture connections: slides 20–25. Instructor-led preview.*

### 6.1 Stemming groups forms using transformation rules
We use **Porter's original algorithm** so that the implementation matches the version identified in the lecture comparison. A stem can be a shortened form that is not a dictionary word. [NLTK Porter stemmer](https://www.nltk.org/api/nltk.stem.porter.html)

**Inspect:** What happens to `organization` and `organ`? Would that grouping help a comparison of organizational reports with medical documents?


In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer(mode=PorterStemmer.ORIGINAL_ALGORITHM)
for word in ["cats", "studies", "organization", "organ"]:
    print(word, "→", stemmer.stem(word))


cats → cat
studies → studi
organization → organ
organ → organ


### 6.2 Load a trained English pipeline
The next examples use the model installed in Setup B. `nlp` will be our name for the loaded pipeline. Calling `nlp(some_text)` creates a **Doc**: the text together with linguistic annotations.

This is separate from the blank, rule-based pipeline used earlier. Keep the original wording and capitalization when passing text to the trained pipeline. The displayed versions help us record which tools produced our output. [spaCy model loading](https://spacy.io/usage/models#usage)


In [ ]:
import nltk
import spacy
import en_core_web_sm

nlp = en_core_web_sm.load()
print("spaCy version:", spacy.__version__)
print("English model version:", nlp.meta["version"])
print("NLTK version:", nltk.__version__)


spaCy version: 3.8.11
English model version: 3.8.0
NLTK version: 3.9.2


### 6.3 Compare a stem with a lemma in context
A **lemma** is a base form such as `study` for the verb `studied`. This English pipeline's lemmatizer uses grammatical information supplied by other components.

`token.text` retrieves the original token; `token.lemma_` retrieves its lemma. The display loop skips punctuation so that we can focus on the words. This display choice leaves the original document unchanged. [spaCy token attributes](https://spacy.io/api/token#attributes)


In [ ]:
morphology_text = "The organizations studied reports about climate change."
morphology_doc = nlp(morphology_text)

print("TOKEN | PORTER STEM | LEMMA")
for token in morphology_doc:
    if not token.is_punct:
        print(token.text, stemmer.stem(token.text), token.lemma_, sep=" | ")


TOKEN | PORTER STEM | LEMMA
The | the | the
organizations | organ | organization
studied | studi | study
reports | report | report
about | about | about
climate | climat | climate
change | chang | change


## **7. Inspect linguistic annotations**
*Lecture connections: slides 26–34. Instructor-led preview.*

### 7.1 Part-of-speech tagging: the two uses of “saw”
The lecture sentence uses `saw` as a verb and as a noun. The model predicts grammatical labels from context.

`token.pos_` gives a broad category such as `VERB` or `NOUN`. In this English model, `token.tag_` gives a finer-grained label such as `VBD` or `NN`, connecting to the Penn Treebank tags in the slides. The underscore is part of each attribute's name. [spaCy POS tagging](https://spacy.io/usage/linguistic-features#pos-tagging)


In [ ]:
pos_text = "John saw the saw and decided to take it to the table."
pos_doc = nlp(pos_text)

print("TOKEN | BROAD POS | FINE-GRAINED TAG")
for token in pos_doc:
    print(token.text, token.pos_, token.tag_, sep=" | ")

print("VBD means:", spacy.explain("VBD"))


TOKEN | BROAD POS | FINE-GRAINED TAG
John | PROPN | NNP
saw | VERB | VBD
the | DET | DT
saw | NOUN | NN
and | CCONJ | CC
decided | VERB | VBD
to | PART | TO
take | VERB | VB
it | PRON | PRP
to | ADP | IN
the | DET | DT
table | NOUN | NN
. | PUNCT | .
VBD means: verb, past tense


### 7.2 Noun phrases: keep meaningful spans together
Use the mouse-and-cheese sentence from the lecture. `doc.noun_chunks` gives base noun phrases using spaCy's dependency parse. This differs from implementing the lecture's hand-written `DT? JJ* NN` pattern; both examples help us inspect phrase spans. [spaCy noun chunks](https://spacy.io/api/doc#noun_chunks)

**Inspect:** Are `The little mouse` and `the fresh cheese` identified as phrases? What would be lost if we studied only their individual words?


In [ ]:
chunk_text = "The little mouse ate the fresh cheese."
chunk_doc = nlp(chunk_text)

print("Noun phrases:")
for phrase in chunk_doc.noun_chunks:
    print(phrase.text)


Noun phrases:
The little mouse
the fresh cheese


### 7.3 Named entities: identify mentions of people, organizations, and places
This invented visit story gives us a short example to inspect. `doc.ents` contains predicted entity spans. `entity.text` gives the words; `entity.label_` gives the predicted type. In this model, `GPE` denotes countries, cities, or states. [spaCy named entities](https://spacy.io/usage/linguistic-features#named-entities)

**Inspect the actual output:** Does each span cover the right words? Does the type fit? A missed entity is a model error to investigate; an empty entity list would not establish that the text contains no real entities.


In [ ]:
ner_text = "Maya Chen visited Microsoft in Seattle on Monday."
entity_doc = nlp(ner_text)

print("Predicted entities:")
for entity in entity_doc.ents:
    print(entity.text, "→", entity.label_, "|", spacy.explain(entity.label_))


Predicted entities:
Maya Chen → PERSON | People, including fictional
Microsoft → ORG | Companies, agencies, institutions, etc.
Seattle → GPE | Countries, cities, states
Monday → DATE | Absolute or relative dates or periods


## **8. Takeaways**

We moved from a string to selected representations and annotations:

**Original text → chosen token boundaries → task-specific normalization → inspected annotations**

Keep the original text available. A corpus can have multiple processed versions for different analyses; record what each version changes.


---
## Additional References Below ·

These examples retain additional operations from the original notebook. They are available for review after class and are not prerequisites for either exercise.

### A. Joining, repetition, and converting a number to text
`+` combines strings without adding spaces. `*` repeats a string. `str()` converts a number to its string representation. Joining words does not automatically restore natural punctuation spacing.


In [ ]:
print("Welcome" + "to" + "LIS" + "501" + "!")
print(" ".join(["Welcome", "to", "LIS", "501", "!"]))
print("abc" * 3)
print("Week " + str(2))


### B. Looking up a method
You do not have to memorize every method. For example, run `help(str.split)` below. In a separate Colab code cell, `text.split?` is another way to request documentation. You can also try typing `text.` to explore available methods.


In [ ]:
help(str.split)


### C. A compact 'initials' exercise version
The original uses a **list comprehension** to repeat an operation in one expression. The ordinary loop below performs the same steps: take the first character of each whitespace-separated name part, collect the characters, then join them.

This version includes middle initials because it uses every name part. It still assumes whitespace is an appropriate way to split the name.


In [ ]:
example_name = "Peter Pan"
initial_letters = []
for part in example_name.split():
    initial_letters.append(part[0])
print("Loop version:", "".join(initial_letters))

compact_initials = "".join([part[0] for part in example_name.split()])
print("Compact version:", compact_initials)


### D. Compare entity predictions before and after lowercasing
This uses the trained model loaded in Section 6. Compare the actual outputs; the predictions may change or stay the same. Preserve the original wording so that you can investigate either result.


In [ ]:
comparison_text = "Maya Chen visited Microsoft in Seattle on Monday."
for version in [comparison_text, comparison_text.lower()]:
    comparison_doc = nlp(version)
    print("\nInput:", version)
    print("Number of predicted entities:", len(comparison_doc.ents))
    for entity in comparison_doc.ents:
        print(entity.text, "→", entity.label_)


### Troubleshooting

| What you see | What to check |
|:--|:--|
| `NameError` | Run the earlier cell that creates the variable. Check spelling and capitalization. |
| `SyntaxError` | Check matching quotation marks, parentheses, and square brackets. |
| `IndentationError` | Keep the spaces before instructions inside a loop or condition. |
| `IndexError` | Check that the list or string contains the item you are selecting. |
| A method is displayed instead of a result | Check for missing parentheses, such as `.split` instead of `.split()`. |
| `ModuleNotFoundError` for NLTK or spaCy | Run Setup A and inspect any installation error. |
| `ModuleNotFoundError` for `en_core_web_sm` | Run Setup B. Restart the runtime if prompted, then rerun earlier cells. |
| `CalledProcessError` during setup | The installation failed. Read the preceding message, check the connection, and rerun the failed setup cell. |

If you change an earlier value, rerun the later cells that use it. After restarting a runtime, rerun setup and then continue from the top. Sections 1–5 and the Porter example use no separately downloaded model; they need only the libraries from Setup A. The lemmatization and annotation previews require Setup B as well.
